In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
all_candidates = pd.read_csv('data/nba_breakout_candidates.csv')

sns.scatterplot(data = all_candidates,x='PIE_change', y='PIE_diff')

In [ ]:
sns.regplot(data=all_candidates, x = 'PIE_change', y = 'PIE_diff')
plt.xlabel('PIE Change')
plt.ylabel('PIE Difference')
plt.title('Breakout Player Analysis')
plt.show()

sns.lmplot(data=all_candidates, x='PIE_change', y='PIE_diff', hue='POSITION_simplified')

In [ ]:
sns.histplot(data=all_candidates, x='PIE_diff', bins=20, kde=True)

In [ ]:
import statsmodels.formula.api as smf

diff_model = smf.ols('PIE_diff ~ PIE_change + PIE_pre + USG_change + AGE_post + MIN_post + avg_wins_3yr + C(POSITION_simplified)', data=all_candidates).fit()
display(diff_model.summary())

# Diagnostic plots
sns.scatterplot(y=diff_model.resid, x=diff_model.fittedvalues)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Fitted Values')
plt.ylabel('Residuals')
plt.title('Residuals vs Fitted Values')
plt.show()

import statsmodels.api as sm
sm.qqplot(diff_model.resid, line='s')
plt.title('Q-Q Plot')
plt.show()

# Calculate VIF for multicollinearity
from statsmodels.stats.outliers_influence import variance_inflation_factor
numerical_predictors = all_candidates[['PIE_change', 'PIE_pre', 'USG_change', 'AGE_post', 'MIN_post', 'avg_wins_3yr']]
vif = [variance_inflation_factor(numerical_predictors.values, i) for i in range(numerical_predictors.shape[1])]
vif_df = pd.DataFrame({'Variable': numerical_predictors.columns, 'VIF': vif})
vif_df

In [ ]:
# Refine model to address multicollinearity
diff_model = smf.ols('PIE_diff ~ PIE_change + AGE_post + C(POSITION_simplified)', data=all_candidates).fit()
display(diff_model.summary())

numerical_predictors_reduced = all_candidates[['PIE_change', 'AGE_post']]
vif_reduced = [variance_inflation_factor(numerical_predictors_reduced.values, i) for i in range(numerical_predictors_reduced.shape[1])]
vif_reduced_df = pd.DataFrame({'Variable': numerical_predictors_reduced.columns, 'VIF': vif_reduced})
vif_reduced_df